# 02 · Baselines: Black-Scholes vs Hull-White vs gradient boosting

Owner: baselines lead.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "..")
from config import *
from src import load, build_dataset, models, evaluate

In [ ]:
pairs = pd.read_parquet(PAIRS_PATH)
train, val, test = (pairs[pairs.split == s] for s in ("train", "val", "test"))
len(train), len(val), len(test)

In [ ]:
hw = models.HullWhite().fit(train)
gbm = models.GBMResidual().fit(train, val=val)
print("Hull-White coefficients:", hw.coef_)

In [ ]:
pairs = evaluate.add_predictions(pairs, {"hw": hw, "gbm": gbm})
tables = evaluate.standard_report(pairs, ["bs", "hw", "gbm"], TABLES_DIR / "baselines")
tables["by_split_type"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, t in zip(axes, ("C", "P")):
    tb = evaluate.gain_table(evaluate.with_buckets(pairs[pairs.split == "test"]).query("type == @t"), ["hw", "gbm"], ["abs_delta_bucket"])
    evaluate.plot_gain_by_bucket(tb, ["hw", "gbm"], f"{t}: test year", ax)
plt.tight_layout(); plt.savefig(FIGURES_DIR / "baselines_gain_by_delta.png", dpi=150)

In [ ]:
# persist predictions so later notebooks can reuse them
pairs.to_parquet(PROCESSED_DIR / "pairs_with_baselines.parquet", index=False)